# Chapter 6.3: Diffusion Models for Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the diffusion process (forward noise, reverse denoise) and its application to recommendation
2. **Implement** DiffRec for collaborative filtering with diffusion models
3. **Explain** DREAM for handling both explicit and implicit feedback via diffusion
4. **Describe** DiffuRec for sequential recommendation and CDDRec for cross-domain scenarios
5. **Build** CF-Diff: conditional diffusion for collaborative filtering
6. **Compare** diffusion-based methods with VAE and GAN approaches
7. **Analyze** the trade-off between diffusion steps and recommendation quality

## Prerequisites

- Familiarity with PyTorch and neural network training
- Understanding of denoising autoencoders
- Chapters 6.1-6.2 (VAE and GAN for recommendation)
- Basic knowledge of diffusion models (DDPM) is helpful but not required

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part6/chapter_6.3_diffusion_rec.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part6/chapter_6.3_diffusion_rec.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import math

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f'Using device: {device}')

## 1. Diffusion Process Recap

### Forward Process (Adding Noise)

Starting from the data $\mathbf{x}_0$, we gradually add Gaussian noise over $T$ timesteps:

$$q(\mathbf{x}_t | \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t; \sqrt{1-\beta_t}\mathbf{x}_{t-1}, \beta_t \mathbf{I})$$

where $\beta_1, ..., \beta_T$ is a noise schedule. We can sample $\mathbf{x}_t$ directly:

$$q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t; \sqrt{\bar{\alpha}_t}\mathbf{x}_0, (1-\bar{\alpha}_t)\mathbf{I})$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$.

### Reverse Process (Denoising)

A neural network $\epsilon_\theta$ learns to predict the noise added at each step:

$$p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1}; \mu_\theta(\mathbf{x}_t, t), \sigma_t^2 \mathbf{I})$$

### Why Diffusion for Recommendation?

1. **Better modeling of uncertainty**: Diffusion naturally captures uncertainty in user preferences
2. **Iterative refinement**: Multi-step denoising allows progressive improvement of predictions
3. **Flexible generation**: Can condition on partial observations

> **💡 Concept:** In recommendation, $\mathbf{x}_0$ is the user's interaction vector. The forward process corrupts it, and the reverse process learns to reconstruct clean preferences from noisy observations.

In [ ]:
# Visualize the forward diffusion process on a simple interaction vector
def get_noise_schedule(T, schedule_type='linear', s=0.008):
    """Get beta schedule for diffusion process."""
    if schedule_type == 'linear':
        betas = np.linspace(1e-4, 0.02, T)
    elif schedule_type == 'cosine':
        steps = np.arange(T + 1) / T
        alphas_bar = np.cos((steps + s) / (1 + s) * np.pi / 2) ** 2
        alphas_bar = alphas_bar / alphas_bar[0]
        betas = 1 - alphas_bar[1:] / alphas_bar[:-1]
        betas = np.clip(betas, 1e-4, 0.999)
    else:
        raise ValueError(f'Unknown schedule: {schedule_type}')
    return betas


# Demonstrate forward process
T = 100
betas = get_noise_schedule(T, 'linear')
alphas = 1 - betas
alphas_bar = np.cumprod(alphas)

# Create a sparse interaction vector
x0 = np.zeros(50)
x0[[3, 7, 12, 18, 25, 31, 40, 45]] = 1.0  # 8 interacted items

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
timesteps_to_show = [0, 10, 25, 50, 75, 99]

for idx, t in enumerate(timesteps_to_show):
    ax = axes[idx // 3, idx % 3]
    if t == 0:
        xt = x0
    else:
        noise = np.random.randn(*x0.shape)
        xt = np.sqrt(alphas_bar[t]) * x0 + np.sqrt(1 - alphas_bar[t]) * noise
    ax.bar(range(50), xt, color='steelblue', alpha=0.7)
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.set_title(f't = {t} (SNR: {alphas_bar[t]/(1-alphas_bar[t]+1e-8):.2f})', fontsize=10)
    ax.set_ylim(-3, 3)
    ax.set_xlabel('Item Index')

plt.suptitle('Forward Diffusion: Corrupting User Interactions', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Synthetic Data

In [ ]:
def generate_synthetic_data(n_users=1500, n_items=300, n_factors=8, sparsity=0.06, seed=42):
    rng = np.random.RandomState(seed)
    user_factors = rng.randn(n_users, n_factors) * 0.5
    item_factors = rng.randn(n_items, n_factors) * 0.5
    logits = user_factors @ item_factors.T + rng.randn(n_items) * 0.2
    probs = 1.0 / (1.0 + np.exp(-logits))
    threshold = np.quantile(probs, 1 - sparsity)
    interactions = (probs > threshold).astype(np.float32)
    
    train_data = interactions.copy()
    test_data = np.zeros_like(interactions)
    for u in range(n_users):
        items = np.where(interactions[u] > 0)[0]
        if len(items) > 1:
            n_test = max(1, int(len(items) * 0.2))
            test_items = rng.choice(items, n_test, replace=False)
            train_data[u, test_items] = 0
            test_data[u, test_items] = 1
    return train_data, test_data

train_data, test_data = generate_synthetic_data()
n_users, n_items = train_data.shape
print(f'Data: {n_users} users x {n_items} items, density={train_data.mean():.4f}')

## 3. DiffRec: Diffusion for Collaborative Filtering

**DiffRec** (Wang et al., SIGIR 2023) applies diffusion models to collaborative filtering. The key idea is to treat the user interaction vector as the data to denoise.

### Architecture

The denoising network $\epsilon_\theta(\mathbf{x}_t, t)$ is typically an MLP with:
- Time embedding via sinusoidal positional encoding
- Residual connections for stable training

### Training Objective

The simplified denoising objective:
$$\mathcal{L} = \mathbb{E}_{t, \mathbf{x}_0, \epsilon} \left[ \|\epsilon - \epsilon_\theta(\mathbf{x}_t, t)\|^2 \right]$$

Or equivalently, predict $\mathbf{x}_0$ directly:
$$\mathcal{L} = \mathbb{E}_{t, \mathbf{x}_0, \epsilon} \left[ \|\mathbf{x}_0 - \hat{\mathbf{x}}_0(\mathbf{x}_t, t)\|^2 \right]$$

> **🔑 Pro Tip:** For recommendation, predicting $\mathbf{x}_0$ directly (rather than noise) often works better because the interaction vectors are sparse and bounded.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal time embedding for diffusion models."""
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        return emb


class DiffRecDenoiser(nn.Module):
    """Denoising network for DiffRec.
    
    Reference: Wang et al., "Diffusion Recommender Model", SIGIR 2023.
    """
    
    def __init__(self, n_items, hidden_dim=256, time_dim=64, n_layers=2):
        super().__init__()
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Input projection
        self.input_proj = nn.Linear(n_items, hidden_dim)
        
        # Residual blocks
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.Sequential(
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, hidden_dim)
            ))
        
        # Output projection
        self.output_proj = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_items)
        )
    
    def forward(self, x_t, t):
        """Predict x_0 from noisy x_t and timestep t."""
        # Time embedding
        t_emb = self.time_embed(t)
        t_emb = self.time_proj(t_emb)
        
        # Input
        h = self.input_proj(x_t) + t_emb
        
        # Residual layers
        for layer in self.layers:
            h = h + layer(h)
        
        return self.output_proj(h)


class DiffRec:
    """DiffRec: Diffusion-based Collaborative Filtering."""
    
    def __init__(self, n_items, T=50, hidden_dim=256, schedule='linear'):
        self.T = T
        self.n_items = n_items
        
        # Noise schedule
        betas = get_noise_schedule(T, schedule)
        self.betas = torch.FloatTensor(betas).to(device)
        self.alphas = 1 - self.betas
        self.alphas_bar = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_bar = torch.sqrt(self.alphas_bar)
        self.sqrt_one_minus_alphas_bar = torch.sqrt(1 - self.alphas_bar)
        
        # Denoising network
        self.model = DiffRecDenoiser(n_items, hidden_dim).to(device)
    
    def q_sample(self, x_0, t, noise=None):
        """Forward process: sample x_t from x_0."""
        if noise is None:
            noise = torch.randn_like(x_0)
        sqrt_ab = self.sqrt_alphas_bar[t].unsqueeze(1)
        sqrt_one_minus_ab = self.sqrt_one_minus_alphas_bar[t].unsqueeze(1)
        return sqrt_ab * x_0 + sqrt_one_minus_ab * noise
    
    def training_loss(self, x_0):
        """Compute training loss (predict x_0)."""
        batch_size = x_0.size(0)
        t = torch.randint(0, self.T, (batch_size,), device=device)
        noise = torch.randn_like(x_0)
        x_t = self.q_sample(x_0, t, noise)
        
        # Predict x_0
        x_0_pred = self.model(x_t, t)
        
        # MSE loss on interaction reconstruction
        loss = F.mse_loss(x_0_pred, x_0)
        return loss
    
    @torch.no_grad()
    def p_sample(self, x_t, t):
        """Reverse process: sample x_{t-1} from x_t."""
        # Predict x_0
        t_batch = torch.full((x_t.size(0),), t, device=device, dtype=torch.long)
        x_0_pred = self.model(x_t, t_batch)
        
        if t > 0:
            # Compute posterior mean
            alpha_t = self.alphas[t]
            alpha_bar_t = self.alphas_bar[t]
            alpha_bar_t_prev = self.alphas_bar[t - 1] if t > 0 else torch.tensor(1.0)
            
            # Posterior variance
            beta_tilde = self.betas[t] * (1 - alpha_bar_t_prev) / (1 - alpha_bar_t)
            
            # Posterior mean
            coef1 = torch.sqrt(alpha_bar_t_prev) * self.betas[t] / (1 - alpha_bar_t)
            coef2 = torch.sqrt(alpha_t) * (1 - alpha_bar_t_prev) / (1 - alpha_bar_t)
            mean = coef1 * x_0_pred + coef2 * x_t
            
            noise = torch.randn_like(x_t)
            return mean + torch.sqrt(beta_tilde) * noise
        else:
            return x_0_pred
    
    @torch.no_grad()
    def sample(self, batch_size=None, x_T=None):
        """Generate predictions by running full reverse process."""
        if x_T is None:
            x_T = torch.randn(batch_size, self.n_items, device=device)
        
        x = x_T
        for t in reversed(range(self.T)):
            x = self.p_sample(x, t)
        return x


print('DiffRec model created.')
diff_rec = DiffRec(n_items=n_items, T=50, hidden_dim=256)
print(f'Parameters: {sum(p.numel() for p in diff_rec.model.parameters()):,}')

## 4. Training DiffRec

In [ ]:
def compute_metrics(scores, train_data, test_data, k=20):
    """Compute Recall@K and NDCG@K."""
    scores[torch.FloatTensor(train_data).to(device) > 0] = -float('inf')
    _, top_k = torch.topk(scores, k, dim=1)
    top_k = top_k.cpu().numpy()
    
    recalls, ndcgs = [], []
    for u in range(test_data.shape[0]):
        true = set(np.where(test_data[u] > 0)[0])
        if len(true) == 0:
            continue
        hits = [1 if i in true else 0 for i in top_k[u]]
        recalls.append(sum(hits) / min(len(true), k))
        dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
        idcg = sum(1.0 / np.log2(j + 2) for j in range(min(len(true), k)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(recalls), np.mean(ndcgs)


def train_diffrec(diff_rec, train_data, test_data, n_epochs=60, batch_size=256, lr=1e-3):
    """Train DiffRec."""
    optimizer = torch.optim.Adam(diff_rec.model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    train_tensor = torch.FloatTensor(train_data).to(device)
    dataset = TensorDataset(train_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    history = {'loss': [], 'recall': [], 'ndcg': []}
    
    for epoch in range(n_epochs):
        diff_rec.model.train()
        epoch_loss = 0
        n_batches = 0
        
        for (batch_x,) in loader:
            loss = diff_rec.training_loss(batch_x)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diff_rec.model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        
        if (epoch + 1) % 10 == 0:
            diff_rec.model.eval()
            # For evaluation, use fewer steps for speed
            with torch.no_grad():
                # Start from noisy version of training data
                t_eval = diff_rec.T - 1
                t_batch = torch.full((len(train_tensor),), t_eval, device=device, dtype=torch.long)
                x_T = diff_rec.q_sample(train_tensor, t_batch)
                
                # Run a few denoising steps
                x = x_T
                step_size = max(1, diff_rec.T // 10)  # Use 10 steps for evaluation
                for t in range(t_eval, -1, -step_size):
                    x = diff_rec.p_sample(x, t)
                
                scores = x
            
            recall, ndcg = compute_metrics(scores.clone(), train_data, test_data)
            history['loss'].append(epoch_loss / n_batches)
            history['recall'].append(recall)
            history['ndcg'].append(ndcg)
            print(f'DiffRec Epoch {epoch+1:3d} | Loss: {epoch_loss/n_batches:.4f} | '
                  f'Recall@20: {recall:.4f} | NDCG@20: {ndcg:.4f}')
    
    return history


diff_rec = DiffRec(n_items=n_items, T=50, hidden_dim=256)
diffrec_history = train_diffrec(diff_rec, train_data, test_data, n_epochs=60)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

eval_epochs = list(range(10, 61, 10))

axes[0].plot(eval_epochs, diffrec_history['loss'], 'b-o', markersize=6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('DiffRec Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_epochs, diffrec_history['recall'], 'g-o', label='Recall@20', markersize=6)
axes[1].plot(eval_epochs, diffrec_history['ndcg'], 'm-s', label='NDCG@20', markersize=6)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Metric')
axes[1].set_title('DiffRec Recommendation Quality')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. DREAM: Diffusion for Explicit and Implicit Feedback

**DREAM** (Diffusion-based REcommendAtion Model) extends DiffRec to handle both explicit ratings and implicit feedback. The key difference:

- For **implicit feedback**: reconstruct binary interaction vectors (as in DiffRec)
- For **explicit feedback**: reconstruct rating vectors, using rating-aware noise

DREAM also introduces **importance-aware diffusion** where the noise schedule is item-dependent, adding less noise to items the user has rated highly.

> **💡 Concept:** The intuition is that high-confidence interactions (explicit high ratings) should be corrupted less, while uncertain interactions get more noise.

In [ ]:
class DREAM(DiffRec):
    """DREAM: Diffusion for explicit/implicit feedback."""
    
    def __init__(self, n_items, T=50, hidden_dim=256):
        super().__init__(n_items, T, hidden_dim)
    
    def importance_weighted_noise(self, x_0, noise):
        """Scale noise inversely with interaction strength."""
        # Items with interactions get less noise
        weight = torch.where(x_0 > 0, torch.tensor(0.5), torch.tensor(1.0))
        return noise * weight
    
    def training_loss(self, x_0, importance_weighting=True):
        batch_size = x_0.size(0)
        t = torch.randint(0, self.T, (batch_size,), device=device)
        noise = torch.randn_like(x_0)
        
        if importance_weighting:
            noise = self.importance_weighted_noise(x_0, noise)
        
        x_t = self.q_sample(x_0, t, noise)
        x_0_pred = self.model(x_t, t)
        
        # Weighted MSE: higher weight on observed items
        weight = torch.where(x_0 > 0, torch.tensor(5.0), torch.tensor(1.0))
        loss = (weight * (x_0_pred - x_0) ** 2).mean()
        return loss


dream = DREAM(n_items=n_items, T=50, hidden_dim=256)
print(f'DREAM model parameters: {sum(p.numel() for p in dream.model.parameters()):,}')

# Quick training
optimizer = torch.optim.Adam(dream.model.parameters(), lr=1e-3)
train_tensor = torch.FloatTensor(train_data).to(device)
dataset = TensorDataset(train_tensor)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

for epoch in range(30):
    dream.model.train()
    total_loss = 0
    for (batch_x,) in loader:
        loss = dream.training_loss(batch_x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'DREAM Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}')

## 6. DiffuRec: Sequential Recommendation with Diffusion

**DiffuRec** applies diffusion to sequential recommendation, where the goal is to predict the next item given a sequence of past interactions.

Key idea: Instead of diffusing the full interaction vector, DiffuRec diffuses the **target item embedding** conditioned on the user's sequence.

$$\mathbf{x}_0 = \text{embed}(i_{\text{target}}), \quad \text{condition} = \text{SeqEncoder}([i_1, ..., i_{t-1}])$$

## 7. CF-Diff: Conditional Diffusion for CF

**CF-Diff** conditions the diffusion process on user features or partial interaction history:

$$\epsilon_\theta(\mathbf{x}_t, t, \mathbf{c})$$

where $\mathbf{c}$ can be user demographics, partial history, or even collaborative signals from similar users.

In [ ]:
class ConditionalDiffRecDenoiser(nn.Module):
    """Conditional denoiser that takes user features as condition."""
    
    def __init__(self, n_items, cond_dim=32, hidden_dim=256, time_dim=64):
        super().__init__()
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU()
        )
        self.cond_proj = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.SiLU()
        )
        self.input_proj = nn.Linear(n_items, hidden_dim)
        
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
                nn.Linear(hidden_dim, hidden_dim)
            ) for _ in range(2)
        ])
        self.output_proj = nn.Linear(hidden_dim, n_items)
    
    def forward(self, x_t, t, cond):
        t_emb = self.time_proj(self.time_embed(t))
        c_emb = self.cond_proj(cond)
        h = self.input_proj(x_t) + t_emb + c_emb
        for layer in self.layers:
            h = h + layer(h)
        return self.output_proj(h)


# Create user features (simulate with PCA of interactions)
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=32, random_state=42)
user_features = svd.fit_transform(train_data)
user_features = torch.FloatTensor(user_features).to(device)

# Train conditional DiffRec
cond_denoiser = ConditionalDiffRecDenoiser(n_items, cond_dim=32, hidden_dim=256).to(device)
optimizer = torch.optim.Adam(cond_denoiser.parameters(), lr=1e-3)

T = 50
betas_t = torch.FloatTensor(get_noise_schedule(T, 'linear')).to(device)
alphas_t = 1 - betas_t
alphas_bar_t = torch.cumprod(alphas_t, dim=0)

for epoch in range(30):
    cond_denoiser.train()
    total_loss = 0
    for i in range(0, n_users, 256):
        batch_x = train_tensor[i:i+256]
        batch_cond = user_features[i:i+256]
        batch_size = batch_x.size(0)
        
        t = torch.randint(0, T, (batch_size,), device=device)
        noise = torch.randn_like(batch_x)
        sqrt_ab = alphas_bar_t[t].sqrt().unsqueeze(1)
        sqrt_one_minus_ab = (1 - alphas_bar_t[t]).sqrt().unsqueeze(1)
        x_t = sqrt_ab * batch_x + sqrt_one_minus_ab * noise
        
        x_0_pred = cond_denoiser(x_t, t, batch_cond)
        loss = F.mse_loss(x_0_pred, batch_x)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f'CF-Diff Epoch {epoch+1} | Loss: {total_loss/(n_users/256):.4f}')

## 8. Diffusion Steps vs Quality Analysis

In [ ]:
# Analyze how number of diffusion steps affects recommendation quality
step_counts = [1, 5, 10, 25, 50]
step_results = {}

diff_rec.model.eval()
for n_steps in step_counts:
    with torch.no_grad():
        t_eval = diff_rec.T - 1
        t_batch = torch.full((len(train_tensor),), t_eval, device=device, dtype=torch.long)
        x = diff_rec.q_sample(train_tensor, t_batch)
        
        step_size = max(1, diff_rec.T // n_steps)
        for t in range(t_eval, -1, -step_size):
            x = diff_rec.p_sample(x, t)
        
        scores = x.clone()
    
    recall, ndcg = compute_metrics(scores, train_data, test_data)
    step_results[n_steps] = {'recall': recall, 'ndcg': ndcg}
    print(f'{n_steps:3d} steps | Recall@20: {recall:.4f} | NDCG@20: {ndcg:.4f}')

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
steps = list(step_results.keys())
recalls = [step_results[s]['recall'] for s in steps]
ndcgs = [step_results[s]['ndcg'] for s in steps]

ax.plot(steps, recalls, 'b-o', label='Recall@20', markersize=8)
ax.plot(steps, ndcgs, 'r-s', label='NDCG@20', markersize=8)
ax.set_xlabel('Number of Diffusion Steps', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('DiffRec: Quality vs Inference Steps', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

> **⚠️ Common Pitfall:** More diffusion steps do not always mean better recommendations. There is a sweet spot where additional steps yield diminishing returns. For production systems, find the minimum number of steps that achieves acceptable quality.

## Exercises

### 🏋️ Exercise 1: Implement DiffRec with Noise Prediction

Modify DiffRec to predict noise $\epsilon$ instead of $\mathbf{x}_0$ and compare performance.

In [ ]:
# TODO: Implement noise-prediction variant of DiffRec
class DiffRecNoisePred(DiffRec):
    def training_loss(self, x_0):
        """
        TODO:
        1. Sample t and noise
        2. Create x_t
        3. Predict noise (not x_0)
        4. MSE loss between predicted and true noise
        """
        pass
    
    def p_sample(self, x_t, t):
        """
        TODO: Recover x_{t-1} from predicted noise
        x_{t-1} = (x_t - sqrt(1-alpha_bar_t) * eps_pred) / sqrt(alpha_t)
        + noise (if t > 0)
        """
        pass

# TODO: Train and compare with x_0 prediction variant

### 🏋️ Exercise 2: Cosine Schedule vs Linear Schedule

Compare cosine and linear noise schedules for DiffRec.

In [ ]:
# TODO: Compare noise schedules
# 1. Train DiffRec with linear schedule
# 2. Train DiffRec with cosine schedule
# 3. Plot the alpha_bar curves for both
# 4. Compare Recall@20 and NDCG@20
# 5. Analyze: which schedule preserves signal longer?

### 🏋️ Exercise 3: Implement Guided Diffusion for Recommendation

Add classifier-free guidance to DiffRec for controlling the diversity-accuracy trade-off.

In [ ]:
# TODO: Implement classifier-free guidance
# Key idea: train with condition dropout, at inference use:
# pred = (1 + w) * pred_conditioned - w * pred_unconditioned
# where w is the guidance scale
#
# 1. Modify training to randomly drop user features (set to zeros) with prob 0.1
# 2. At inference, compute both conditioned and unconditioned predictions
# 3. Sweep guidance scale w = [0, 0.5, 1.0, 2.0, 5.0]
# 4. Plot accuracy vs diversity for different w values

## Summary

| Model | Key Innovation | Application | Year |
|-------|---------------|-------------|------|
| **DiffRec** | Diffusion for interaction vectors | Collaborative filtering | 2023 |
| **DREAM** | Importance-weighted noise | Explicit + implicit feedback | 2023 |
| **DiffuRec** | Diffusion on item embeddings | Sequential recommendation | 2023 |
| **CDDRec** | Cross-domain diffusion | Cross-domain recommendation | 2023 |
| **CF-Diff** | Conditional diffusion | Feature-enhanced CF | 2023 |

**Key Takeaways:**
1. Diffusion models offer a principled way to model uncertainty in user preferences
2. Predicting $\mathbf{x}_0$ directly works better than noise prediction for sparse recommendations
3. The number of inference steps controls the quality-speed trade-off
4. Conditioning on user features or partial history improves performance